# Part 7 · Notebook 06 — Directional option strategies and the structure selector

**Sessions:** S6 (Directional & mean-reversion option strategies) · [Lesson plan](../../docs/lessons/PART_07_STRATEGY_LIBRARY.md) · graded labs in [`labs/part07/`](../../labs/part07/)

**You will:**
1. Pick strikes by delta.
2. Compute IV rank and IV percentile, and see how they differ.
3. Price a combo at its mid and at its natural price.
4. Compare three ways to be bullish, and map a view to a structure.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic, built from regimes you know, and every strategy here is a **hypothesis** with a first-look evaluation: the honest backtest comes in Part 8.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p7lib.py is in notebooks/part07/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p7lib as p

p.use_course_style()

In [ ]:
S, T = 600.0, 30 / 365
ivf = p.skew_iv(S)
strikes = np.arange(450, 751, 5.0)

## 1. Strikes by delta

Traders ask for "the 30-delta call" or "the 16-delta put", not a strike. Price each listed strike with its own IV (`ivf(K)`), compute its delta with `p.bsm_greeks(S, K, T, r, q, iv, cp)["delta"]`, and return the strike whose delta is closest to the target (puts have negative deltas).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def strike_for_delta(S, T, r, q, iv_fn, cp, target, strikes):
    K = np.asarray(strikes, dtype=float)
    d = ...                                       # ✍️ the delta of every strike, each with its own IV
    return float(K[np.argmin(np.abs(d - target))])

targets = [(1, 0.50), (1, 0.30), (1, 0.16), (-1, -0.30), (-1, -0.16)]
mine = [p.attempt(strike_for_delta, S, T, 0.04, 0.0, ivf, cp, tg, strikes) for cp, tg in targets]
mine = p.check("strike_for_delta", mine, [p.strike_for_delta(S, T, 0.04, 0.0, ivf, cp, tg, strikes) for cp, tg in targets])
dict(zip(["50Δ call", "30Δ call", "16Δ call", "30Δ put", "16Δ put"], mine))

The 16-delta put is further from spot than the 16-delta call: the skew makes puts expensive, so their deltas are larger at the same distance.

## 2. Is implied vol high?

Over the last `window` values (today included):
* **IV rank** = `(today − min) / (max − min) × 100` (50 if flat): where today sits in the range;
* **IV percentile** = share (%) of the previous `window − 1` values strictly **below** today: how often it was lower.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def iv_rank(hist, window=252):
    x = np.asarray(hist, dtype=float)[-window:]
    return ...                                    # ✍️

def iv_percentile(hist, window=252):
    x = np.asarray(hist, dtype=float)[-window:]
    return ...                                    # ✍️

hist = p.iv_history()
days = [300, 420, 599]
mine = [(p.attempt(iv_rank, hist[: d + 1]), p.attempt(iv_percentile, hist[: d + 1])) for d in days]
mine = p.check("iv rank and percentile", mine, [(p.iv_rank(hist[: d + 1]), p.iv_percentile(hist[: d + 1])) for d in days])
fig, ax = plt.subplots(figsize=(10, 3.2)); ax.plot(hist * 100); [ax.axvline(d, color=p.PALETTE[7], ls="--", lw=1) for d in days]
ax.set_title("30-day implied vol, %"); plt.show()
pd.DataFrame(mine, index=[f"day {d}" for d in days], columns=["IV rank", "IV percentile"]).round(1)

One spike in the window drags the rank down for a year; the percentile ignores how big the spike was. Say which one a rule uses.

## 3. Combo prices: mid vs natural

A multi-leg order is quoted per share as a net price (positive = you pay a debit). The **mid** is `Σ qty·(bid + ask)/2`. The **natural** price is what crossing every leg costs: buy at the ask, sell at the bid, `Σ (qty·ask if qty > 0 else qty·bid)`. `leg_cost = natural − mid` is what legging in would throw away.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def combo_prices(qtys, bids, asks):
    q, b, a = (np.asarray(v, dtype=float) for v in (qtys, bids, asks))
    mid = float(np.sum(q * (b + a) / 2))
    natural = ...                                 # ✍️
    return {"mid": mid, "natural": natural, "leg_cost": natural - mid}

condor = dict(qtys=[1, -1, -1, 1], bids=[2.10, 4.05, 3.60, 1.45], asks=[2.25, 4.25, 3.80, 1.60])
spread = dict(qtys=[1, -1], bids=[13.90, 5.30], asks=[14.30, 5.55])
mine = [p.attempt(combo_prices, **condor), p.attempt(combo_prices, **spread)]
mine = p.check("combo_prices", mine, [p.combo_prices(**condor), p.combo_prices(**spread)])
pd.DataFrame(mine, index=["iron condor (credit)", "bull call spread (debit)"]).round(3)

The condor's credit at the mid is $4.15 per share; crossing all four legs gives up $0.35 of it, about 8% of the premium, on every entry and again on every exit. Send one combo order at (or near) the net mid.

## 4. Three ways to be bullish

100 shares, one 600 call, or a 600/620 bull call spread: same direction, very different shapes, costs and Greeks.

In [ ]:
ways = {"100 shares": p.OptionStrategy("stock").add(0, 0, 0, 1),
        "long 600 call": p.OptionStrategy("call").add(1, 600, T, 1, ivf(600)),
        "600/620 bull call spread": p.vertical(1, 600, 620, T, ivf)}
g = np.linspace(540, 660, 300)
fig, ax = plt.subplots()
rows = {}
for k, st in ways.items():
    ax.plot(g, st.value(g, dt=T) - st.value(S), label=k)
    has_options = any(L.cp != 0 for L in st.legs)
    a = st.analyze(S) if has_options else {"cost": float(st.value(S)), "max_loss": -float(st.value(S))}   # stock can go to 0
    gr = st.greeks(S)
    rows[k] = {"cost $": a["cost"], "max loss $": a["max_loss"], "delta (shares)": gr["delta"], "vega $/pt": gr["vega"], "theta $/day": gr["theta"]}
ax.set_ylim(-3000, 3000); ax.axhline(0, color="black", lw=0.6); ax.set(xlabel="price at expiry", ylabel="P&L, $"); ax.legend(); plt.show()
pd.DataFrame(rows).T.round(1)

## 5. From a view to a structure

`p.select_structure` writes down the classic rules as code: with a directional view, buy options when IV is low, use debit spreads in the middle, sell credit spreads when IV is rich; with no view, sell a condor when IV is high, a calendar in contango, else no trade. It is a **hypothesis** to test in Part 8, not a law.

In [ ]:
grid = pd.DataFrame({ivr: {d: p.select_structure(d, ivr, term_slope=0.01) for d in (1, 0, -1)} for ivr in (10, 45, 85)})
grid.index = ["bullish", "no view", "bearish"]; grid.columns = [f"IV rank {c}" for c in grid.columns]
grid

## Wrap-up

* Strikes by delta, on a smile; IV rank vs percentile, stated explicitly.
* Always send multi-leg trades as one combo at the net mid; model the leg cost in first looks.
* Graded version: `labs/part07/week24_option_builder` and the Clinic W2 option playbook.